# Tunisia Socioeconomic Dashboard: Map Visualization

This notebook creates an **interactive map** showing poverty rates and school dropout rates per delegation in Tunisia.

In [ ]:
import pandas as pd
import geopandas as gpd
import folium

csv_path = "../data/clean/tunisia_poverty_2015_cleaned.csv"
df = pd.read_csv(csv_path)

shapefile_path = "../data/raw/TUN_adm1.shp"
gdf = gpd.read_file(shapefile_path, encoding='ISO-8859-1')

gdf = gdf.to_crs(epsg=4326)

In [ ]:
import unidecode

messy_names = ['Ariana', 'BA(c)ja', 'Ben Arous (Tunis Sud)', 'Bizerte', 'GabA"s', 'Gafsa',
               'Jendouba', 'Kairouan', 'KassA(c)rine', 'Kebili', 'Le Kef', 'Mahdia', 'Manubah',
               'MA(c)denine', 'Monastir', 'Nabeul', 'Sfax', 'Sidi Bou Zid', 'Siliana',
               'Sousse', 'Tataouine', 'Tozeur', 'Tunis', 'Zaghouan']

clean_names = ['Ariana', 'Beja', 'Ben Arous', 'Bizerte', 'Gabes', 'Gafsa',
               'Jendouba', 'Kairouan', 'Kasserine', 'Kebili', 'Kef', 'Mahdia', 'Manouba',
               'Medenine', 'Monastir', 'Nabeul', 'Sfax', 'Sidi Bouzid', 'Siliana',
               'Sousse', 'Tataouine', 'Tozeur', 'Tunis', 'Zaghouan']

# --- Create mapping dictionary ---
name_mapping = dict(zip(messy_names, clean_names))

# --- Replace NAME_1 with cleaned names ---
gdf['Governorate'] = gdf['NAME_1'].apply(lambda x: unidecode.unidecode(x))
gdf['Governorate'] = gdf['Governorate'].map(name_mapping)

# Check the result
print(gdf['Governorate'].unique())

In [ ]:
merged = gdf.merge(df, on='Governorate')
# missing = merged[merged['Poverty%'].isna()]
# if len(missing) > 0:
#     print("Warning: Some governorates did not match the CSV:")
#     print(missing[['Governorate']])


In [ ]:
m = folium.Map(location=[34.0, 9.0], zoom_start=6, tiles='cartodbpositron')

folium.Choropleth(
    geo_data=merged,
    data=merged,
    columns=['Governorate', 'Poverty%'],
    key_on='feature.properties.Governorate',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name='Poverty rate (%)'
).add_to(m)

# Optional: Add popups showing values
for _, row in merged.iterrows():
    folium.GeoJson(
        row['geometry'],
        tooltip=folium.Tooltip(f"{row['Governorate']}: {row['Poverty%']}%"),
        style_function=lambda x: {'fillOpacity': 0, 'color': 'black', 'weight': 0.5}  # only border
    ).add_to(m)


# Display map
# m.save("tunisia_poverty_map.html")

m


In [ ]:
# Now we will do the map by delegation
print(gdf['NAME_2'].unique())

merged_delegation = gdf.merge(df, left_on='NAME_2', right_on='Delegation')

In [ ]:
from rapidfuzz import fuzz, process

# Lists of names
shapefile_names = gdf['NAME_2'].tolist()

french_to_english = {
    "Sud": "South",
    "Ville": "City",
    "Ouest": "West",
    "Est": "East",
    "Nord": "North",
    "Centre": "Center",
    "Nouvelle": "New",
    "Superieur": "Superior",
}



shapefile_names = [
    ' '.join([french_to_english.get(word, word) for word in name.split()])
    for name in shapefile_names
]

# I wanna remove unknown, unkown1, Lake Ichkeul from shapefile names
shapefile_names = [name for name in shapefile_names if name not in ['Unknown', 'Unknown1', 'Lake Ichkeul']]

csv_names = df['Delegation'].tolist()

#Let us print how many delegations are in the csv and shapefile
print(f"Number of delegations in shapefile: {len(shapefile_names)}")
print(f"Number of delegations in CSV: {len(csv_names)}")

# Create a dictionary to map shapefile name -> best CSV match
mapping = {}
for name in shapefile_names:
    best_match = process.extractOne(name, csv_names, scorer=fuzz.token_sort_ratio)
    mapping[name] = best_match[0]  # best_match[0] is the matched CSV name


# Let us print the mapping for each values to check
# Let us sort by the shapefile names for easier reading
mapping = dict(sorted(mapping.items()))

for k, v in mapping.items():
    print(f"'{k}' -> '{v}'")

In [ ]:
# Add a new column with cleaned CSV-compatible delegation names
gdf['Delegation_clean'] = gdf['NAME_2'].map(mapping)
merged_delegation = gdf.merge(df, left_on='Delegation_clean', right_on='Delegation')

In [ ]:
m_deleg = folium.Map(location=[34.0, 9.0], zoom_start=6, tiles='cartodbpositron')
folium.Choropleth(
    geo_data=merged_delegation,
    data=merged_delegation,
    columns=['Delegation', 'Poverty%'],
    key_on='feature.properties.Delegation',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name='Poverty rate (%) by Delegation'
).add_to(m_deleg)

folium.GeoJson(
    merged_delegation,
    tooltip=folium.GeoJsonTooltip(fields=['Delegation', 'Poverty%'],
                                   aliases=['Delegation:', 'Poverty Rate (%):']),
    style_function=lambda x: {'fillOpacity': 0, 'color': 'black', 'weight': 0.5}  # only border
).add_to(m_deleg)

# Display map
# m_deleg.save("tunisia_poverty_map_delegation.html")
m_deleg

In [ ]:
from rapidfuzz import fuzz, process

# Example names
shapefile_names = ["Beja Ville", "Ariana", "Ben Arous"]
csv_names = ["Beja City", "Ariena", "BenArous"]

# Check fuzzy match score for each pair
for name in shapefile_names:
    best_match = process.extractOne(name, csv_names, scorer=fuzz.token_sort_ratio)
    print(f"'{name}' best matches with '{best_match[0]}' with score {best_match[1]}")


In [ ]:
# Could we just print the map with the NAME_2 without any mapping, just to see the ones that do not match?

# So I can see each delegation name in the shapefile with its geographical location in the real map

# --- IGNORE ---

folium.Map(location=[34.0, 9.0], zoom_start=6, tiles='cartodbpositron')
for _, row in gdf.iterrows():
    folium.GeoJson(
        row['geometry'],
        tooltip=folium.Tooltip(f"{row['NAME_2']}"),
        style_function=lambda x: {'fillOpacity': 0, 'color': 'black', 'weight': 0.5}  # only border
    ).add_to(m)
m